# FarrahAI — Notebook 4: Question Paper Prediction

This notebook demonstrates:
- Building teacher profiles from past question papers
- XGBoost-based topic importance prediction
- Top-k accuracy evaluation
- Generating a predicted sample paper
- Topic frequency visualization


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from modules.teacher_profile import (
    load_teacher_db, save_teacher_db,
    add_paper_to_teacher, get_teacher_summary,
    get_top_topics
)
from modules.ml_models import build_topic_importance_dataset, train_importance_predictor
from modules.predictor import generate_sample_paper, format_paper_output

TEACHER_DB_PATH = '../data/teacher_profiles.json'
print('Ready ✓')

## Step 1: Build Teacher Profiles from Past Papers

Add your actual teacher's past paper data here.
Each paper is a JSON object with topics and marks.


In [ ]:
# Load or create DB
db = load_teacher_db(TEACHER_DB_PATH)

# ── Simulate Prof. Sharma's past papers ──────────────────────────────────
# Replace these with your actual teacher's real past papers!

papers_sharma = [
    {
        'subject': 'AI_ML', 'paper_type': 'endsem', 'year': 2022, 'semester': 'odd',
        'total_marks': 100,
        'topics': [
            {'name': 'Neural Networks', 'marks': 10, 'question': 'Explain backpropagation'},
            {'name': 'Supervised Learning', 'marks': 5, 'question': 'Compare SVM and LR'},
            {'name': 'Model Evaluation', 'marks': 5, 'question': 'Define F1 score'},
            {'name': 'Decision Trees', 'marks': 10, 'question': 'Build a decision tree'},
            {'name': 'Clustering', 'marks': 5, 'question': 'K-means algorithm steps'},
        ]
    },
    {
        'subject': 'AI_ML', 'paper_type': 'endsem', 'year': 2023, 'semester': 'odd',
        'total_marks': 100,
        'topics': [
            {'name': 'Neural Networks', 'marks': 10, 'question': 'Derive backprop equations'},
            {'name': 'Ensemble Methods', 'marks': 10, 'question': 'Explain XGBoost'},
            {'name': 'Model Evaluation', 'marks': 5, 'question': 'ROC curve explanation'},
            {'name': 'Reinforcement Learning', 'marks': 10, 'question': 'Q-learning algorithm'},
            {'name': 'Deep Learning', 'marks': 5, 'question': 'CNN architecture'},
        ]
    },
    {
        'subject': 'AI_ML', 'paper_type': 'internal', 'year': 2023, 'semester': 'odd',
        'total_marks': 30,
        'topics': [
            {'name': 'Supervised Learning', 'marks': 5, 'question': 'Types of supervised learning'},
            {'name': 'Neural Networks', 'marks': 10, 'question': 'MLP architecture'},
            {'name': 'Model Evaluation', 'marks': 5, 'question': 'Precision recall tradeoff'},
        ]
    },
    {
        'subject': 'AI_ML', 'paper_type': 'endsem', 'year': 2024, 'semester': 'odd',
        'total_marks': 100,
        'topics': [
            {'name': 'Neural Networks', 'marks': 15, 'question': 'LSTM architecture'},
            {'name': 'Model Evaluation', 'marks': 10, 'question': 'Cross validation types'},
            {'name': 'Ensemble Methods', 'marks': 10, 'question': 'Random forest vs XGBoost'},
            {'name': 'Clustering', 'marks': 5, 'question': 'Compare K-means DBSCAN'},
            {'name': 'Deep Learning', 'marks': 10, 'question': 'Transfer learning methods'},
        ]
    },
]

for paper in papers_sharma:
    add_paper_to_teacher(db, 'Prof_Sharma', paper)

save_teacher_db(db, TEACHER_DB_PATH)
print('\n── Teacher Profile ──────────────────────────────────')
print(get_teacher_summary('Prof_Sharma', db))

## Step 2: Topic Frequency Visualization

In [ ]:
top_topics = get_top_topics('Prof_Sharma', db, top_n=10)
topics     = [t['topic'] for t in top_topics]
counts     = [t['count'] for t in top_topics]

colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(topics)))[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(topics[::-1], counts[::-1], color=colors)
ax.bar_label(bars, fmt='%d', padding=4)
ax.set_xlabel('Times Asked in Past Papers')
ax.set_title('Topic Frequency — Prof. Sharma (AI/ML)')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../outputs/topic_frequency_chart.png', dpi=150)
plt.show()

## Step 3: XGBoost — Topic Importance Prediction

In [ ]:
# Create a labeled dataset for training
# 'is_important' = 1 if topic appeared in actual exam, 0 otherwise
# You manually label this from your ground truth papers
importance_data = [
    {'topic': 'Neural Networks',       'frequency': 4, 'total_marks': 45, 'recency_score': 0.95, 'appeared_in_internal': True,  'appeared_in_endsem': True,  'is_important': 1},
    {'topic': 'Model Evaluation',      'frequency': 4, 'total_marks': 25, 'recency_score': 0.90, 'appeared_in_internal': True,  'appeared_in_endsem': True,  'is_important': 1},
    {'topic': 'Ensemble Methods',      'frequency': 2, 'total_marks': 20, 'recency_score': 0.85, 'appeared_in_internal': False, 'appeared_in_endsem': True,  'is_important': 1},
    {'topic': 'Supervised Learning',   'frequency': 2, 'total_marks': 10, 'recency_score': 0.60, 'appeared_in_internal': True,  'appeared_in_endsem': False, 'is_important': 1},
    {'topic': 'Clustering',            'frequency': 2, 'total_marks': 10, 'recency_score': 0.70, 'appeared_in_internal': False, 'appeared_in_endsem': True,  'is_important': 1},
    {'topic': 'Deep Learning',         'frequency': 2, 'total_marks': 15, 'recency_score': 0.80, 'appeared_in_internal': False, 'appeared_in_endsem': True,  'is_important': 1},
    {'topic': 'Reinforcement Learning','frequency': 1, 'total_marks': 10, 'recency_score': 0.50, 'appeared_in_internal': False, 'appeared_in_endsem': True,  'is_important': 0},
    {'topic': 'Decision Trees',        'frequency': 1, 'total_marks': 10, 'recency_score': 0.30, 'appeared_in_internal': False, 'appeared_in_endsem': True,  'is_important': 0},
    {'topic': 'Dimensionality Reduc.', 'frequency': 0, 'total_marks': 0,  'recency_score': 0.10, 'appeared_in_internal': False, 'appeared_in_endsem': False, 'is_important': 0},
    {'topic': 'Bayesian Methods',      'frequency': 0, 'total_marks': 0,  'recency_score': 0.05, 'appeared_in_internal': False, 'appeared_in_endsem': False, 'is_important': 0},
    {'topic': 'Optimization',          'frequency': 1, 'total_marks': 5,  'recency_score': 0.40, 'appeared_in_internal': True,  'appeared_in_endsem': False, 'is_important': 0},
    {'topic': 'NLP Basics',            'frequency': 0, 'total_marks': 0,  'recency_score': 0.05, 'appeared_in_internal': False, 'appeared_in_endsem': False, 'is_important': 0},
]

df_imp = build_topic_importance_dataset(importance_data)
model_imp, metrics_imp = train_importance_predictor(df_imp)

from modules.ml_models import save_model
save_model(model_imp, '../models/importance_predictor_xgboost.pkl')
print('\nImportance predictor saved ✓')

## Step 4: Feature Importance (XGBoost)

In [ ]:
feature_names = ['frequency', 'total_marks', 'recency_score',
                 'appeared_in_internal', 'appeared_in_endsem']
importances   = model_imp.feature_importances_

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feature_names, importances, color='#3498db')
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_title('XGBoost Feature Importance — Topic Importance Prediction')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../outputs/xgboost_feature_importance.png', dpi=150)
plt.show()

## Step 5: Generate Sample Predicted Paper

In [ ]:
paper = generate_sample_paper(
    teacher_name='Prof_Sharma',
    subject='AI_ML',
    teacher_db=db,
    total_marks=100,
    paper_type='endsem'
)

print(format_paper_output(paper))

with open('../outputs/predicted_paper_AI_ML_Prof_Sharma.json', 'w') as f:
    json.dump(paper, f, indent=2)
print('Predicted paper saved ✓')

## Step 6: Top-k Accuracy Evaluation

In [ ]:
# Evaluate how many of our predicted top-k topics actually appeared
# in a held-out real exam paper

# Replace 'actual_topics' with topics from a real exam paper you have
actual_topics = {'Neural Networks', 'Model Evaluation', 'Deep Learning', 'Clustering'}

predicted_ranked = [t['topic'] for t in get_top_topics('Prof_Sharma', db, top_n=10)]

print('── Top-k Accuracy Evaluation ──────────────────────────')
print(f'Actual topics in exam: {actual_topics}\n')

for k in [3, 5, 7, 10]:
    top_k_set = set(predicted_ranked[:k])
    hits = top_k_set & actual_topics
    precision = len(hits) / k
    recall    = len(hits) / len(actual_topics)
    print(f'Top-{k}: predicted={list(top_k_set)[:k]}')
    print(f'        Hits: {hits}')
    print(f'        Precision@{k}: {precision:.2f}  Recall@{k}: {recall:.2f}\n')